# Safe Autonomy Tutorial 2: Motion Planning and Hamilton-Jacobi Reachability

## C-level, Hamilton-Jacobi reachability

In [ ]:
# Pin hj_reachability to the API used in this notebook.
!pip install -q hj-reachability==0.7.0 shapely cvxopt plotly

In [ ]:
import os, math, time

os.environ.setdefault("JAX_PLATFORMS", "cpu")   # Default to CPU unless JAX_PLATFORMS is already set.

import numpy as np
import jax, jax.numpy as jnp
import hj_reachability as hj
import shapely
from shapely.geometry import Polygon as ShapelyPolygon, Point
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from matplotlib.patches import Polygon, Circle
from matplotlib.collections import PatchCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# The same map and robot as in the E-level notebook.
obstacles = [np.array(o) for o in [   # polygon vertices [x, y] in m, one list per obstacle
    [[2.633, 1.267], [3.033, 1.867], [3.833, 0.867]],
    [[1.575, 3.862], [1.175, 4.462], [2.375, 4.862], [2.375, 4.062]],
    [[3.75, 3.55], [4.95, 4.35], [5.55, 3.75], [4.75, 3.35]],
    [[7.35, 3.25], [7.95, 4.05], [7.95, 2.45]],
    [[6.238, 0.662], [5.638, 1.262], [6.038, 1.862], [6.838, 1.462]],
    [[4.217, 6.3], [5.017, 7.1], [6.017, 6.1]],
]]
x0 = np.array([1.25, 1.25, 0.785398])   # initial state [x, y, theta], in m and rad
xG = np.array([7.9, 6.6, 0.0])          # goal state [x, y, theta], in m and rad
MAP_W = 8.75          # map width, in m
MAP_H = 7.5           # map height, in m
ROBOT_RADIUS = 0.2    # the robot is overapproximated by a disk of this radius, in m
GOAL_RADIUS = 0.3     # radius of the goal disk, in m
V_MIN = 0.0           # speed bounds, in m/s
V_MAX = 1.0
W_MAX = 1.0           # turn-rate bound, in rad/s
D_MAX = 0.1           # disturbance bound per axis, in m/s
DT = 0.1              # time step of the E-level plan, in s
print(f"map {MAP_W} x {MAP_H} m, robot radius {ROBOT_RADIUS} m, goal radius {GOAL_RADIUS} m, "
      f"v in [{V_MIN}, {V_MAX}] m/s, |w| <= {W_MAX} rad/s, disturbance <= {D_MAX} m/s per axis")
print("jax backend:", jax.default_backend())


### C0. Where the E-level left off

The E-level trajectory can fail under bounded disturbances. To deal with this, we look at Hamilton-Jacobi reachability in this tutorial, with the same map, robot and disturbance bound as in the E-level.

The disturbance $\mathbf{d} = [d_x, d_y]^T$ lies in the box $\mathcal{D} = \{[d_x, d_y]^T \mid |d_x| \le d_{\max},\ |d_y| \le d_{\max}\}$. Hamilton-Jacobi analysis [1] identifies the initial states from which the robot can satisfy a requirement despite this disturbance.

We use $t$ for physical time and $T$ for the terminal time. The remaining horizon length is $T-t$. We consider three problems.

| Problem | Robot's objective despite disturbance | Section |
|---|---|---|
| **Reach** | Reach the goal at $T$, or at some time during $[t,T]$ | C3 |
| **Avoid** | Avoid obstacles throughout $[t,T]$ | C4 |
| **Reach-avoid** | Reach the goal at $T$ while avoiding obstacles throughout $[t,T]$ | C5 |

Each problem is characterized by a value function $V(t,\mathbf{x})$. The set $\{\mathbf{x} \mid V(t,\mathbf{x}) \le 0\}$ contains the states from which a suitable control strategy satisfies the requirement despite the disturbance. The framework `hj_reachability` uses the level set method of the lecture, storing $V$ on a grid and computing it backward in time from a terminal condition. We formulate a problem in four steps.

1. Dynamics in control- and disturbance-affine form (C1).
2. Functions representing the goal and obstacles on a grid (C2).
3. A value function and solver settings for the chosen requirement (C3 to C5).
4. Feedback controller from the value function (A-level).

Here we define the unicycle dynamics in NumPy and an angle-wrapping helper, both used throughout the notebook.

In [ ]:
def unicycle_f(x, u, d):
    return np.array([u[0] * math.cos(x[2]) + d[0], u[0] * math.sin(x[2]) + d[1], u[1]])

def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


### C1. Modeling the robot for the solver

`hj_reachability` supports general nonlinear dynamics, but in this tutorial we only look at dynamics which are affine in the control input and disturbance,

$$
\dot{\mathbf{x}} = \mathbf{f}_0(\mathbf{x}) + \mathbf{G}_u(\mathbf{x})\, \mathbf{u} + \mathbf{G}_d(\mathbf{x})\, \mathbf{d},
$$

with the drift $\mathbf{f}_0$, the control Jacobian $\mathbf{G}_u$ and the disturbance Jacobian $\mathbf{G}_d$. As an example, we use the continuous-time unicycle model again:

$$
\begin{bmatrix}
\dot x \\
\dot y \\
\dot\theta
\end{bmatrix} =
\begin{bmatrix}
v\cos\theta + d_x \\
v\sin\theta + d_y \\
\omega
\end{bmatrix}.
$$

This model has zero drift. The state is $\mathbf{x} = [x, y, \theta]^T$, with position $\mathbf{p} = [x, y]^T$. The control input is $\mathbf{u} = [v, \omega]^T \in \mathcal{U} = \{[v, \omega]^T \mid 0 \le v \le v_{\max},\ |\omega| \le \omega_{\max}\}$, with speed $v$ and turn rate $\omega$ (`w` in the code). The disturbance is $\mathbf{d} = [d_x, d_y]^T \in \mathcal{D}$. All bounds are inherited from the E-level. 


**Exercise C1.** Complete the three methods of `Unicycle`: `open_loop_dynamics` returns the drift $\mathbf{f}_0$, `control_jacobian` returns $\mathbf{G}_u$, and `disturbance_jacobian` returns $\mathbf{G}_d$.


*Hints.*

- Each method receives one state as a JAX array of shape `(3,)`.
- Use `jnp` functions, since `hj_reachability` traces the methods with JAX.

In [ ]:
class Unicycle(hj.ControlAndDisturbanceAffineDynamics):

    def __init__(self, v_min, v_max, w_max, d_max, control_mode="min", disturbance_mode="max"):
        control_space = hj.sets.Box(jnp.array([v_min, -w_max]), jnp.array([v_max, w_max]))
        disturbance_space = hj.sets.Box(jnp.array([-d_max, -d_max]), jnp.array([d_max, d_max]))
        super().__init__(control_mode, disturbance_mode, control_space, disturbance_space)

    def open_loop_dynamics(self, state, time):
        # BEGIN SOLUTION
        return None   # drift f_0, shape (3,)
        # END SOLUTION

    def control_jacobian(self, state, time):
        # BEGIN SOLUTION
        return None   # control Jacobian G_u, shape (3, 2)
        # END SOLUTION

    def disturbance_jacobian(self, state, time):
        # BEGIN SOLUTION
        return None   # disturbance Jacobian G_d, shape (3, 2)
        # END SOLUTION

# Check against the numpy dynamics at random states, control inputs and disturbances.
rng = np.random.default_rng(0)
dyn_check = Unicycle(V_MIN, V_MAX, W_MAX, D_MAX)
for _ in range(20):
    x = rng.uniform([0, 0, -np.pi], [MAP_W, MAP_H, np.pi]); u = rng.uniform([V_MIN, -W_MAX], [V_MAX, W_MAX]); d = rng.uniform(-D_MAX, D_MAX, 2)
    assert np.allclose(dyn_check(jnp.array(x), jnp.array(u), jnp.array(d), 0.), unicycle_f(x, u, d), atol=1e-5)
print("Unicycle matches the numpy dynamics")

We represent $V$ on a grid over the map in $x$ and $y$ and over the full circle in $\theta$, which we declare periodic. A finer grid resolves thin sets better but increases memory use and computation time, and can require smaller solver time steps.

The cell below sets the grid resolution and two time steps: `DT_VALUE`, the spacing in time at which the solver stores $V$, and `DT_SIM`, the time step of the rollout simulations.

In [ ]:
GRID_SHAPE = (71, 61, 51)                  # (nx, ny, ntheta)
DT_VALUE = 0.25                        # spacing of the stored value slices, in s
DT_SIM = 0.05                          # rollout time step, in s

In [ ]:
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    hj.sets.Box(np.array([0., 0., -np.pi]), np.array([MAP_W, MAP_H, np.pi])), GRID_SHAPE, periodic_dims=2)
XY = np.asarray(grid.states[:, :, 0, :2])           # shape (nx, ny, 2)
THETAS = np.asarray(grid.coordinate_vectors[2])
print(f"grid {GRID_SHAPE}, {np.prod(GRID_SHAPE):,} nodes, node spacing {[round(float(s), 3) for s in grid.spacings]} in x [m], y [m] and theta [rad]")

In [ ]:
#@title Plotting helpers for value arrays of shape (nx, ny, ntheta) { display-mode: "form" }

def heading_index(theta):
    return int(np.argmin(np.abs(THETAS - theta)))

def heading_slice(values, theta=None, reduce=np.min):
    # theta=None reduces over all headings.
    v = np.asarray(values)
    return v[:, :, heading_index(theta)] if theta is not None else reduce(v, axis=2)

def start_pose(theta):
    return (x0[0], x0[1], theta)

def draw_obstacles(ax, polygons=None):
    ax.add_collection(PatchCollection([Polygon(o) for o in (obstacles if polygons is None else polygons)], color=[1, 0, 0], alpha=0.6))

def heading_arrow(ax, x, y, theta, color="k", length=0.6, width=0.006, **kw):
    ax.quiver(x, y, np.cos(theta), np.sin(theta), color=color, angles="xy", scale_units="xy", scale=1 / length, width=width, **kw)

def map_axes(ax=None, title=None, polygons=None, show_obstacles=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6 * MAP_H / MAP_W))
    if show_obstacles:
        draw_obstacles(ax, polygons)
    ax.add_patch(Circle(xG[:2], GOAL_RADIUS, color="g", alpha=0.4))
    ax.add_patch(Circle(x0[:2], ROBOT_RADIUS, color="b", alpha=0.3)); ax.plot(x0[0], x0[1], "bo")
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H); ax.set_aspect("equal")
    ax.set_xlabel("x in [m]"); ax.set_ylabel("y in [m]")
    if title:
        ax.set_title(title)
    return ax

def plot_heading(ax, traj, every=10, color="k"):
    heading_arrow(ax, traj[0, ::every], traj[1, ::every], traj[2, ::every], color=color)

def draw_robot(ax, theta, pos=None):
    pos = x0[:2] if pos is None else pos
    ax.add_patch(Circle(pos, ROBOT_RADIUS, color="b", alpha=0.3)); ax.plot(pos[0], pos[1], "bo")
    heading_arrow(ax, pos[0], pos[1], theta, length=0.7, width=0.008, zorder=5)

def plot_set(values, ax, theta=None, reduce=np.min, color="k", fill=False, label=None):
    # Draw V = 0 and optionally fill V <= 0.
    v2 = heading_slice(values, theta, reduce) if np.ndim(values) == 3 else np.asarray(values)
    if fill:
        ax.contourf(XY[..., 0], XY[..., 1], v2, levels=[-1e9, 0], colors=[color], alpha=0.3)
    ax.contour(XY[..., 0], XY[..., 1], v2, levels=[0], colors=[color], linewidths=1.5)
    if label:
        ax.plot([], [], color=color, label=label)

def animate_sets(panels, frames, frame_title, robot_at=None, polygons_at=None, path=None, color="#2171b5", show_obstacles=True):
    # Each panel is (label, values_fn), with values_fn(frame) returning a 2D value array.
    # robot_at(frame) returns (x, y, theta), polygons_at(frame) returns polygons, and path has shape (3, N).
    fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5.5 * MAP_H / MAP_W + 0.4), squeeze=False)
    def draw(i):
        f = frames[i]
        for ax, (label, values_fn) in zip(axes[0], panels):
            ax.cla()
            map_axes(ax, title=f"{label}, {frame_title(f)}", polygons=None if polygons_at is None else polygons_at(f), show_obstacles=show_obstacles)
            plot_set(values_fn(f), ax, color=color, fill=True)
            if robot_at is not None:
                x, y, theta = robot_at(f)
                draw_robot(ax, theta, pos=np.array([x, y]))
            if path is not None:
                ax.plot(path[0], path[1], "k-", lw=0.8, alpha=0.5)
        return []
    anim = FuncAnimation(fig, draw, frames=len(frames), interval=250, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def plot_inputs(ax, t, U, title, step=False):
    draw = (lambda *a, **k: ax.step(*a, where="post", **k)) if step else ax.plot
    draw(t, U[0], "b.-" if step else "b-", label="speed v in [m/s]")
    draw(t, U[1], "g.-" if step else "g-", label="turn rate w in [rad/s]")
    for b in (V_MIN, V_MAX, -W_MAX, W_MAX):
        ax.axhline(b, color="k", lw=0.6, ls=":")
    ax.set_xlabel("time in [s]"); ax.set_title(title); ax.legend(fontsize=8); ax.grid(True)

### C2. Modeling the sets as functions

We represent a target set as $\mathcal{T}=\{\mathbf{x} \mid l(\mathbf{x})\le0\}$, and use this representation for both the goal and obstacles. A convenient choice of $l$ is the signed distance to the boundary $\partial\mathcal{T}$

$$
s_{\mathcal{T}}(\mathbf{x}) = \begin{cases} -\operatorname{dist}(\mathbf{x}, \partial\mathcal{T}) & \mathbf{x} \in \mathcal{T} \\ \phantom{-}\operatorname{dist}(\mathbf{x}, \partial\mathcal{T}) & \mathbf{x} \notin \mathcal{T} \end{cases}.
$$

Consider two sets $\mathcal{T}_1 = \{\mathbf{x} \mid l_1(\mathbf{x}) \le 0\}$ and $\mathcal{T}_2 = \{\mathbf{x} \mid l_2(\mathbf{x}) \le 0\}$. Their union $\mathcal{T}_1 \cup \mathcal{T}_2$ is represented by $\min(l_1, l_2)$, since a point is inside when it is inside either set. Their intersection $\mathcal{T}_1 \cap \mathcal{T}_2$ is represented by $\max(l_1, l_2)$, since a point is inside only when it is inside both. The complement of $\mathcal{T}_1$ is represented by $-l_1$, which swaps inside and outside. The figure illustrates these operations.

<figure>
  
</figure>

| Set | Definition | Function $l$ |
|---|---|---|
| Goal $\mathcal{G}$ | Disk of radius $\rho_G$ around $\mathbf{p}_G$ | $l_{\mathcal{G}}(\mathbf{x}) = \|\mathbf{p} - \mathbf{p}_G\| - \rho_G$ |
| Obstacles $\mathcal{O}$ | Polygons $\mathcal{P}_i$ and the map boundary, inflated by the robot radius $r$ | $l_{\mathcal{O}}(\mathbf{x}) = \min(\min_i s_{\mathcal{P}_i}(\mathbf{p}), s_{\mathrm{boundary}}(\mathbf{p})) - r$ |

Subtracting the robot radius $r$ inflates the obstacles, so collision checks can use the robot centre. We also treat the region outside the map as an obstacle, so the robot cannot leave the map. $s_{\mathrm{boundary}}(\mathbf{p})$ is the distance from $\mathbf{p}$ to the nearest map edge, positive inside the map and negative outside.


**Exercise C2.** Compute `sd_goal` and `sd_boundary` on the position grid `XY`, then complete `compute_sd_obs`, which builds the obstacle function from the polygons and `sd_boundary`.

*Hints.*

- `XY` holds the position $\mathbf{p}$ of every grid node, shape `(nx, ny, 2)`, from the grid cell. All three arrays below have shape `(nx, ny)`.
- `sd_goal`: $\mathbf{p}_G$ is `xG[:2]` and $\rho_G$ is `GOAL_RADIUS`.
- `sd_boundary`: the map is the rectangle with `0 <= x <= MAP_W` and `0 <= y <= MAP_H`, so the distance to the nearest edge is the minimum of four distances.
- `compute_sd_obs`: fill in the signed distance $s_{\mathcal{P}}$ to one polygon. Note that it has different signs inside and outside the polygon. `shapely.contains_xy` tells which nodes lie inside. Outside, use `shapely.distance(poly, pts)`. Inside, use `shapely.distance(poly.exterior, pts)`, since the distance to the filled polygon is zero there.

In [ ]:
def compute_sd_obs(polygons, XY, sd_boundary):
    # Obstacle function l_O at every node of XY: signed distance to the inflated obstacles and map boundary, negative inside.
    # polygons: list of corner-point arrays, each of shape (n_vertices, 2) with the corners in order, e.g. `obstacles`.
    # XY: positions of the grid nodes, shape (nx, ny, 2).
    # sd_boundary: distance to the nearest map edge at every node, positive inside the map, shape (nx, ny).
    pts = shapely.points(XY.reshape(-1, 2))                  # the grid nodes as Shapely points, flat array of length nx * ny
    sd_polygons = []
    for vertices in polygons:
        poly = ShapelyPolygon(vertices)
        # BEGIN SOLUTION
        is_inside = None   # which points lie inside poly, boolean array of length nx * ny
        sd = None          # s_P(p): distance to the polygon boundary, mind the sign, array of length nx * ny
        # END SOLUTION
        sd_polygons.append(sd.reshape(XY.shape[:2]))
    return np.minimum(np.min(sd_polygons, axis=0), sd_boundary) - ROBOT_RADIUS   # l_O = min(min_i s_Pi, s_boundary) - r

# BEGIN SOLUTION
sd_goal = None       # l_G at every node of XY, shape (nx, ny)
sd_boundary = None   # s_boundary at every node of XY, shape (nx, ny)
sd_obs = compute_sd_obs(obstacles, XY, sd_boundary)
# END SOLUTION

assert sd_goal.shape == XY.shape[:2] and sd_obs.shape == XY.shape[:2]
# The solver stores values on the full grid, so we repeat the two functions along the heading axis.
l_goal = jnp.broadcast_to(jnp.array(sd_goal)[:, :, None], GRID_SHAPE)
l_obs = jnp.broadcast_to(jnp.array(sd_obs)[:, :, None], GRID_SHAPE)
assert float(grid.interpolate(l_goal, jnp.array([*xG[:2], 0.]))) < 0 and float(grid.interpolate(l_obs, jnp.array([*x0[:2], 0.]))) > 0
assert float(grid.interpolate(l_obs, jnp.array([*obstacles[0].mean(0), 0.]))) < 0

ax = map_axes(title="Zero level sets of the functions l of the goal and of the obstacles")
plot_set(l_goal, ax, theta=0., color="g", label="goal")
plot_set(l_obs, ax, theta=0., color="k", label="obstacles and the boundary inflated by r")
ax.legend(); plt.show()

### C3. Reaching the goal

Here the target is the goal, with $l=l_{\mathcal{G}}$, and we ignore obstacles. We compare reaching the target at $T$ with reaching it at some time during $[t,T]$.

The trajectory starts at $\mathbf{x}$ at time $t$, and $\Phi(s)$ is its state at a later time $s$. It follows $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d})$ under control input and disturbance signals $\mathbf{u}(\cdot)$ and $\mathbf{d}(\cdot)$, with $\mathbf{u} \in \mathcal{U}$ and $\mathbf{d} \in \mathcal{D}$ for all $s$. The BRS and BRT differ in when the target must be reached.

| | Backward reachable set (BRS) | Backward reachable tube (BRT) |
|---|---|---|
| Target requirement | Be in the target at terminal time $T$ | Be in the target at some time during $[t,T]$ |
| Value function | $V(t, \mathbf{x}) = \min_{\mathbf{u}(\cdot)} \max_{\mathbf{d}(\cdot)} \, l(\Phi(T))$ | $V(t, \mathbf{x}) = \min_{\mathbf{u}(\cdot)} \max_{\mathbf{d}(\cdot)} \min_{s \in [t, T]} l(\Phi(s))$ |
| Set | $\{\mathbf{x} \mid V(t, \mathbf{x}) \le 0\}$ | $\{\mathbf{x} \mid V(t, \mathbf{x}) \le 0\}$ |
| Terminal-value problem | **Hamilton-Jacobi-Isaacs (HJI)**<br>$\partial V / \partial t + H = 0$<br>$V(T, \mathbf{x}) = l(\mathbf{x})$ | **HJI variational inequality (HJI-VI)**<br>$\partial V / \partial t + \min\{0, H\} = 0$<br>$V(T, \mathbf{x}) = l(\mathbf{x})$ |
| Expanded form | $\partial V / \partial t + \min_{\mathbf{u} \in \mathcal{U}} \max_{\mathbf{d} \in \mathcal{D}} \nabla V \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d}) = 0$ | $\partial V / \partial t + \min\{0,\ \min_{\mathbf{u} \in \mathcal{U}} \max_{\mathbf{d} \in \mathcal{D}} \nabla V \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d})\} = 0$ |

Here $H(\mathbf{x}, \nabla V) = \min_{\mathbf{u} \in \mathcal{U}} \max_{\mathbf{d} \in \mathcal{D}} \nabla V \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d})$ is the **Hamiltonian**. $\nabla V$ is the gradient with respect to the state.

Neither terminal-value problem has a closed-form solution in general. We compute the goal BRS and BRT numerically with and without disturbance.

`hj_reachability` solves backward from time $0$ to $-T$, so we shift physical time $t$ to solver time $t_s = -T + t$, placing the terminal time at $t_s=0$. The solver stores the shifted value

$$
\widehat V(t_s,\mathbf{x}) = V(T+t_s,\mathbf{x}).
$$

For a 3 s horizon, `times_reach` runs from `0` to `-3`. The first returned slice is the goal condition, and the last describes the full horizon.

**Exercise C3.** Compute the goal BRS and BRT over 3 s, each with and without disturbance. Before solving, predict which sets contain the others and how disturbance changes them. Compare your predictions with the results.

*Hints.*

- Use `Unicycle`, `hj.SolverSettings`, and `solve(settings, dynamics, times_reach, l_goal)` with the options below.

| Choice | Where to configure it |
|---|---|
| BRS | `hj.SolverSettings.with_accuracy("medium")` uses the default Hamiltonian |
| BRT | Add `hamiltonian_postprocessor=hj.solver.backwards_reachable_tube` to those settings to replace $H$ by $\min\{0,H\}$ |
| Without disturbance | Set `d_max=0.0` in `Unicycle`, the Hamilton-Jacobi-Bellman (HJB) case of lecture 5 |
| With disturbance | Set `d_max=D_MAX` in `Unicycle`, the HJI case |


In [ ]:
def solve(settings, dynamics, times, terminal_values, label=""):
    # Wait for JAX computation to finish before reporting wall-clock time.
    t0 = time.time()
    values = hj.solve(settings, dynamics, grid, times, terminal_values, progress_bar=False)
    values.block_until_ready()
    print(f"{label or 'solve'}: horizon {-times[-1]:.0f} s, {len(times) - 1} stored slices, {time.time() - t0:.1f} s wall clock")
    return values

T_REACH = 3.0
times_reach = np.linspace(0., -T_REACH, int(round(T_REACH / DT_VALUE)) + 1)   # one stored slice every DT_VALUE up to 3 s

# BEGIN SOLUTION
dyn_reach = None     # Unicycle without disturbance
dyn_reach_d = None   # Unicycle with disturbance
settings_set = hj.SolverSettings.with_accuracy(...)                                   # BRS
settings_tube = hj.SolverSettings.with_accuracy(..., hamiltonian_postprocessor=...)   # BRT
V_brs = None     # BRS without disturbance
V_brt = None     # BRT without disturbance
V_brs_d = None   # BRS with disturbance
V_brt_d = None   # BRT with disturbance
# END SOLUTION

def contained(V_small, V_big, tol=0.02):
    # Check containment of zero sublevel sets, allowing value tolerance tol.
    return bool(np.all(np.asarray(V_big)[np.asarray(V_small) <= 0] <= tol))

assert contained(V_brs[-1], V_brt[-1]), "the tube contains the set"
assert contained(V_brt_d[-1], V_brt[-1]), "the disturbance shrinks the reach tube"
assert contained(V_brs_d[-1], V_brs[-1]), "the disturbance shrinks the reach set"

THETA_SLICE = float(x0[2])                                   # default heading for set plots
J_SLICE = heading_index(THETA_SLICE)
heading_frames = [(J_SLICE + k) % len(THETAS) for k in range(0, len(THETAS), 2)]   # sweep over the heading, starting at the start heading
animate_sets([("BRT with disturbance at T = 3 s", lambda j: np.asarray(V_brt_d[-1])[:, :, j])], heading_frames,
             lambda j: f"heading {math.degrees(THETAS[j]):.0f} deg", show_obstacles=False, robot_at=lambda j: start_pose(THETAS[j]))

We now fix the heading to that of the start state, $45^\circ$ in this map, and run the four sets over the horizon.

In [ ]:
panels = [(label, lambda k, V=V: heading_slice(V[k], THETA_SLICE)) for label, V in
          [("BRS, no disturbance", V_brs), ("BRT, no disturbance", V_brt), ("BRS, with disturbance", V_brs_d), ("BRT, with disturbance", V_brt_d)]]
animate_sets(panels, list(range(len(times_reach))), lambda k: f"T = {-times_reach[k]:.2f} s, heading {math.degrees(THETA_SLICE):.0f} deg", show_obstacles=False, robot_at=lambda k: start_pose(THETA_SLICE))

> **Reflection:** Use the animation to answer the following questions and explain your reasoning.
>
> 1. For the same horizon, does the BRS contain the BRT, does the BRT contain the BRS, or do they coincide? How does adding disturbance affect each set?
> 2. Suppose a state belongs to the BRT for a horizon $T$. Does it still belong for every longer horizon? How about the BRS?

### C4. Avoiding the obstacles

The **finite-horizon safe set** is the states from which the robot can keep $g(\Phi(s)) \le 0$ throughout $[t,T]$, despite the disturbance. Here $g = -l_{\mathcal{O}}$ is positive inside the inflated obstacles, zero on their boundaries, and negative outside. The code stores it as `g_obs`.

The safety value function is

$$
V_{\mathrm{safe}}(t,\mathbf{x}) = \min_{\mathbf{u}(\cdot)} \max_{\mathbf{d}(\cdot)} \max_{s\in[t,T]} g(\Phi(s)).
$$

The maximum over time checks the obstacle constraint along the entire trajectory, and the safe set is $\{\mathbf{x} \mid V_{\mathrm{safe}}(t,\mathbf{x}) \le 0\}$. Unlike the BRT in C3, where the target must be reached at some time, the obstacle constraint here must hold at every time. This gives the HJI-VI terminal-value problem

$$
\begin{cases}
\max\left\{\dfrac{\partial V_{\mathrm{safe}}}{\partial t} + H(\mathbf{x},\nabla V_{\mathrm{safe}}),\ g - V_{\mathrm{safe}}\right\} = 0, & t<T,\\
V_{\mathrm{safe}}(T,\mathbf{x}) = g(\mathbf{x}).
\end{cases}
$$

The Hamiltonian $H$ is the same as in C3, with the robot minimizing and the disturbance maximizing. To understand this HJI-VI, we can see that the term $g-V_{\mathrm{safe}}$ enforces $V_{\mathrm{safe}}\ge g$. Therefore, $V_{\mathrm{safe}}\le0$ implies $g\le0$.

**Exercise C4.** Compute the safe sets over 3 s with and without disturbance. Explain how the disturbance changes the safe set and compare this with its effect on the goal BRT.

*Hints.*

- Set `value_postprocessor=hj.solver.static_obstacle(g_obs)` in `hj.SolverSettings.with_accuracy("medium", ...)`.
- Reuse `dyn_reach` and `dyn_reach_d` from C3.
- Use `times_reach` from C3 for the same 3 s horizon.


In [ ]:
g_obs = -l_obs   

# BEGIN SOLUTION
settings_safe = None   # solver settings enforcing the obstacle constraint
V_safe = None          # safe values over 3 s without disturbance
V_safe_d = None        # safe values over 3 s with disturbance
# END SOLUTION

assert contained(V_safe_d[-1], V_safe[-1]), "the disturbance shrinks the safe set"
assert np.all(np.asarray(V_safe) >= np.asarray(g_obs) - 1e-6), "the safe value respects the obstacle constraint"
assert np.all(np.asarray(V_safe_d) >= np.asarray(g_obs) - 1e-6), "the disturbed safe value respects the obstacle constraint"

animate_sets([("safe set, no disturbance", lambda j: np.asarray(V_safe[-1])[:, :, j]), ("safe set, with disturbance", lambda j: np.asarray(V_safe_d[-1])[:, :, j])],
             heading_frames, lambda j: f"T = 3 s, heading {math.degrees(THETAS[j]):.0f} deg", color="#1f77b4", robot_at=lambda j: start_pose(THETAS[j]))

> **Reflection:** Suppose a state lies in the safe set for a horizon $T$. Does the robot can also reach the goal within $T$ from that state?

### C5. Reaching the goal while avoiding the obstacles

The robot should **reach the goal at time $T$ and avoid obstacles throughout the trajectory**. Using the same obstacle function $g$ as in C4, we look at the reach-avoid value introduced in [2]:

$$
V(t, \mathbf{x}) = \min_{\mathbf{u}(\cdot)} \max_{\mathbf{d}(\cdot)} \; \max\Big\{ l_{\mathcal{G}}\big(\Phi(T)\big),\ \max_{s \in [t, T]} g\big(\Phi(s)\big) \Big\}.
$$

The first term checks the goal at $T$. The second checks obstacles along the whole trajectory. Their maximum is nonpositive only when both requirements hold. Thus $V \le 0$ defines the reach-avoid set.

The corresponding terminal-value problems use physical time $t$ with terminal time $T$ and the same reach Hamiltonian $H$ as C3.

| | Goal BRS from C3 | Safe set from C4 | Reach-avoid set |
|---|---|---|---|
| Requirement | Reach the goal at $T$ | Avoid obstacles throughout $[t,T]$ | Reach the goal at $T$ and avoid obstacles throughout the trajectory |
| Terminal-value problem | **HJI**<br>$\partial V/\partial t + H = 0$<br>$V(T,\mathbf{x}) = l_{\mathcal{G}}(\mathbf{x})$ | **HJI-VI**<br>$\max\{\partial V/\partial t + H,\ g - V\} = 0$<br>$V(T,\mathbf{x}) = g(\mathbf{x})$ | **HJI-VI**<br>$\max\{\partial V/\partial t + H,\ g - V\} = 0$<br>$V(T,\mathbf{x}) = \max\{l_{\mathcal{G}}(\mathbf{x}), g(\mathbf{x})\}$ |

**Exercise C5.** Compute the reach-avoid value `V_ra` with disturbance over `T_RA` seconds, and `v_start`, its value at the start state `x0` for the full horizon.

*Hints.*

- Reuse `g_obs` and the obstacle constraint from C4, and `dyn_reach_d` from C3, with the terminal condition in the table.
- `x0` is not a grid node. `grid.interpolate(values, state)` linearly interpolates a grid array at any state.

In [ ]:
T_RA = 13.0   # horizon of the reach-avoid problem, in s
times_ra = np.linspace(0., -T_RA, int(round(T_RA / DT_VALUE)) + 1)

# BEGIN SOLUTION
settings_ra = None   # solver settings with the obstacle postprocessor
V_ra = None          # reach-avoid set with disturbance, one slice per stored time
v_start = None       # value of V_ra at x0 for the full horizon T_RA, a float
# END SOLUTION

assert v_start <= 0, "the start state should be inside the reach-avoid set at the full horizon"
print(f"value at the start state for horizon {T_RA:.0f} s: {v_start:+.2f}")

ax = map_axes(title=f"reach-avoid set with disturbance at T = {T_RA:.0f} s")
plot_set(V_ra[-1], ax, theta=THETA_SLICE, color="#2171b5", fill=True, label=f"heading {math.degrees(THETA_SLICE):.0f} deg (filled)")
draw_robot(ax, THETA_SLICE)
ax.legend(loc="lower right", fontsize=8); plt.show()

> **Reflection:** 
>
> 1. For the same horizon and disturbance bound, could we obtain the reach-avoid set by intersecting the goal BRS from C3 with the safe set from C4?
> 2. How could you estimate the shortest time in which the robot can reach the goal from `x0` safely? 
> 3. Why does the terminal condition of the reach-avoid problem is $\max\{l_{\mathcal{G}}, g\}$ rather than $l_{\mathcal{G}}$ alone?


## A-level, Controller synthesis from the value function

The C-level produced value functions that characterize different sets. In this part, we turn the reach-avoid value function into feedback controllers and test them under the worst disturbance.

The hidden cell below loads the E-level plan `plan_nominal` from `plan_nominal.npy`, saved by the E-level notebook, for comparison with the controllers of this level. It also defines helpers that check a rollout for goal entry and obstacle clearance.

In [ ]:
#@title Rollout helpers: E-level plan, goal entry and obstacle clearance { display-mode: "form" }
PLAN_FILE = "plan_nominal.npy"   # saved at the end of the E-level notebook
assert os.path.exists(PLAN_FILE), "plan_nominal.npy not found: run the E-level notebook first, it saves the plan next to itself"
plan_nominal = np.load(PLAN_FILE)   # rows 0-2 states, rows 3-4 control inputs (the last one is repeated), one column per DT
inputs_plan = plan_nominal[3:, :-1]

def clearance(traj, polygons_per_step):
    return min(ShapelyPolygon(poly).distance(Point(traj[0, k], traj[1, k]))
               for k in range(traj.shape[1]) for poly in polygons_per_step[k])

def static_clearance(traj):
    return clearance(traj, [obstacles] * traj.shape[1])

def reached_goal(X):
    return bool(np.any(np.linalg.norm(X[:2].T - xG[:2], axis=1) <= GOAL_RADIUS))

def rollout_summary(X, dt):
    # Report goal entry, total rollout duration, and minimum center-to-obstacle clearance.
    clr = static_clearance(X)
    return f"reaches goal {reached_goal(X)!s:5s} after {(X.shape[1] - 1) * dt:.1f} s, min clearance {clr:.2f} m{' (collision)' if clr < ROBOT_RADIUS else ''}"


### A1. From value to controller

Along a trajectory with $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x},\mathbf{u},\mathbf{d})$, the chain rule gives

$$
\frac{d}{dt}V(t,\mathbf{x}(t)) = \frac{\partial V}{\partial t} + \nabla V \cdot \mathbf{f}(\mathbf{x},\mathbf{u},\mathbf{d}).
$$

At a fixed state and time, $\partial V/\partial t$ does not depend on the instantaneous control input. The controller therefore minimizes $\nabla V \cdot \mathbf{f}$ against the worst disturbance

$$
\mathbf{u}^*(t,\mathbf{x}) = \arg\min_{\mathbf{u} \in \mathcal{U}} \max_{\mathbf{d} \in \mathcal{D}} \nabla V(t, \mathbf{x}) \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d}).
$$

We use the reach-avoid value `V_ra` from C5 and the disturbed dynamics `dyn_reach_d` from C3. Physical time $t$ corresponds to solver time $-T+t$. The rollout stops at the first goal entry, so an early arrival is not checked at $T$.

**Exercise A1.** Complete `hj_policy` and the forward Euler step in `rollout_closed_loop`. Compare the closed-loop trajectory and control inputs with the E-level open-loop plan loaded above.

*Hints.*

- `optimal_control_and_disturbance` ([definition](https://github.com/StanfordASL/hj_reachability/blob/v0.7.0/hj_reachability/dynamics.py#L71)) ignores its `time` argument for our time-invariant dynamics, so pass `time_dyn`.


In [ ]:
# Spatial gradient of the reach-avoid value, grad V = [dV/dx, dV/dy, dV/dtheta], at every grid node and stored time slice.
GRADS_ra = jnp.stack([grid.grad_values(V_ra[time_idx]) for time_idx in range(len(times_ra))])   # shape (len(times_ra), nx, ny, ntheta, 3), GRADS_ra[time_idx] belongs to V_ra[time_idx]
GRID_LO = jnp.array([0., 0., -jnp.inf])
GRID_HI = jnp.array([MAP_W, MAP_H, jnp.inf])
time_dyn = 0.   # `time` argument for time-varying dynamics; our unicycle ignores it

def clip_to_grid(state):
    # grid.interpolate returns nan outside the grid, so we clip the position to the map. 
    return jnp.clip(jnp.array(state), GRID_LO, GRID_HI)

def hj_policy(dyn, values, grads, times, state, t, T):
    # Control input, worst disturbance and value at `state` and physical time `t`, for a value function solved over horizon T.
    # BEGIN SOLUTION
    time_idx = None   # index of the stored time slice closest to solver time -T + t
    grad_value = None   # grad V interpolated at `state` from grads[time_idx], shape (3,)
    u, d = dyn.optimal_control_and_disturbance(state, time_dyn, grad_value)
    return None, None, None   # control input (2,), worst disturbance (2,), value (scalar)
    # END SOLUTION

def rollout_closed_loop(dyn, values, grads, times, x_start, T, dt):
    # Closed-loop rollout of hj_policy under the worst disturbance. Stop at first goal entry. The remaining motion up to T is not simulated.
    n = int(T / dt)
    X = np.zeros((3, n + 1)); U = np.zeros((2, n)); Vt = np.zeros(n + 1)
    X[:, 0] = x_start
    for k in range(n):
        # BEGIN SOLUTION
        u, d_worst, Vt[k] = hj_policy(dyn, values, grads, times, clip_to_grid(X[:, k]), k * dt, T)   # control input, worst disturbance and value at the current state and time
        U[:, k] = u
        X[:, k + 1] = None                   # forward Euler step with unicycle_f from C0 and the worst disturbance d_worst
        # END SOLUTION
        if np.linalg.norm(X[:2, k + 1] - xG[:2]) <= GOAL_RADIUS:
            Vt[k + 1] = hj_policy(dyn, values, grads, times, clip_to_grid(X[:, k + 1]), (k + 1) * dt, T)[2]
            return X[:, :k + 2], U[:, :k + 1], Vt[:k + 2]
    Vt[n] = hj_policy(dyn, values, grads, times, clip_to_grid(X[:, n]), n * dt, T)[2]
    return X, U, Vt

def rollout_open_loop_worst_disturbance(x_start, inputs, dt):
    # Apply the worst disturbance, selected from the HJ value gradient, along the open-loop plan.
    X = np.zeros((3, inputs.shape[1] + 1)); X[:, 0] = x_start
    for k in range(inputs.shape[1]):
        _, d_worst, _ = hj_policy(dyn_reach_d, V_ra, GRADS_ra, times_ra, clip_to_grid(X[:, k]), k * dt, T_RA)
        X[:, k + 1] = X[:, k] + dt * unicycle_f(X[:, k], inputs[:, k], d_worst)
    return X

X_worst, U_worst, V_worst = rollout_closed_loop(dyn_reach_d, V_ra, GRADS_ra, times_ra, x0, T_RA, DT_SIM)
X_open = rollout_open_loop_worst_disturbance(x0, inputs_plan, DT)

for name, X, dt in [("closed loop, worst disturbance", X_worst, DT_SIM), ("open-loop plan, worst disturbance", X_open, DT)]:
    print(f"{name:33s} {rollout_summary(X, dt)}")
assert reached_goal(X_worst) and static_clearance(X_worst) >= ROBOT_RADIUS - 0.02


In [ ]:
#@title Plot the closed-loop trajectory and the control inputs of the HJ controller { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[1.1, 1]))
map_axes(axes[0], title="closed loop from the reach-avoid value function, worst disturbance")
plot_set(V_ra[-1], axes[0], reduce=np.min, color="#08306b")
axes[0].plot(X_open[0], X_open[1], "r-", label="open-loop plan"); plot_heading(axes[0], X_open, every=10, color="r")
axes[0].plot(X_worst[0], X_worst[1], "b-", label="HJ controller"); plot_heading(axes[0], X_worst, every=20, color="b")
axes[0].legend(fontsize=8)
T_WIN = (4.0, 5.0)
k0, k1 = int(T_WIN[0] / DT_SIM), int(T_WIN[1] / DT_SIM)
plot_inputs(axes[1], np.arange(k0, k1) * DT_SIM, U_worst[:, k0:k1], f"control inputs of the HJ controller between {T_WIN[0]:.0f} and {T_WIN[1]:.0f} s, dotted lines at the bounds", step=True)
plt.tight_layout(); plt.show()


In [ ]:
#@title Animate the value surface along the closed-loop rollout { display-mode: "form" }
# Animate the value surface at the robot's current heading and remaining horizon.
# The reach-avoid set lies at or below the gray V = 0 plane. The robot marker shows its value.
XY2, zero = XY[::2, ::2], np.zeros(XY[::2, ::2].shape[:2])                     # every second node keeps the animation small
angles = np.linspace(0, 2 * np.pi, 60)
goal_circle = xG[:2, None] + GOAL_RADIUS * np.array([np.cos(angles), np.sin(angles)])
def rollout_traces(k):
    i_t = int(np.argmin(np.abs(times_ra - (-T_RA + k * DT_SIM))))            # slice with the remaining horizon T - t
    V2 = np.round(heading_slice(V_ra[i_t], wrap(X_worst[2, k]))[::2, ::2], 2)
    return [go.Surface(x=XY2[..., 0], y=XY2[..., 1], z=V2, colorscale="RdBu_r", cmin=-0.5, cmax=3.0, opacity=0.95, colorbar=dict(title="V", len=0.5, y=0.75), name="V"),
            go.Surface(x=XY2[..., 0], y=XY2[..., 1], z=zero, colorscale=[[0, "gray"], [1, "gray"]], showscale=False, opacity=0.35, name="V = 0"),
            go.Scatter3d(x=goal_circle[0], y=goal_circle[1], z=np.zeros(len(angles)), mode="lines", line=dict(color="green", width=4), name="goal disk"),
            go.Scatter3d(x=X_worst[0, :k + 1], y=X_worst[1, :k + 1], z=V_worst[:k + 1], mode="lines", line=dict(color="black", width=4), name="trajectory at its value"),
            go.Scatter3d(x=[X_worst[0, k]], y=[X_worst[1, k]], z=[V_worst[k]], mode="markers", marker=dict(size=6, color="black"), name="robot at its value")]
frame_ids = sorted(set(range(0, X_worst.shape[1], int(1.0 / DT_SIM))) | {X_worst.shape[1] - 1})   # every second and the last step
fig = go.Figure(data=rollout_traces(frame_ids[0]), frames=[go.Frame(data=rollout_traces(k), name=f"{k * DT_SIM:.0f}") for k in frame_ids])
fig.update_layout(title="reach-avoid value of the remaining horizon on the slice of the robot's heading, closed loop under the worst disturbance",
                  scene=dict(xaxis_title="x in [m]", yaxis_title="y in [m]", zaxis_title="V in [m]", aspectmode="manual",
                             aspectratio=dict(x=1, y=MAP_H / MAP_W, z=0.5), zaxis=dict(range=[-0.5, 3.0])),
                  width=800, height=620, margin=dict(l=0, r=0, t=40, b=0), legend=dict(x=0.01, y=0.98),
                  updatemenus=[dict(type="buttons", x=0.02, y=0.02, buttons=[dict(label="play", method="animate", args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True)])])],
                  sliders=[dict(x=0.15, y=0.02, len=0.8, currentvalue=dict(prefix="t = ", suffix=" s"), font=dict(size=10),
                                steps=[dict(method="animate", label=f.name, args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for f in fig.frames])])
fig.show()

The plot shows that the controller only uses the bounds of the control input set, full or zero speed and full turn rate to one side. We refer to such a control input as **bang-bang**.

> **Reflection:** Why does the control input only take values at the bounds? (Hint: look at the expression of $\mathbf{u}^*$) 

### A2. A reach-avoid safety filter from the value function

To reduce the bang-bang behavior in A1, we introduce a reach-avoid safety filter that adjusts a reference control input only as much as needed to satisfy a constraint. Following [3], we use the reach-avoid value for static obstacles from C5 as a time-varying control barrier function (CBF)

$$
h(t, \mathbf{x}) = -V(t,\mathbf{x}) = -\widehat V(-T+t,\mathbf{x}),
$$

so $h \ge 0$ identifies states from which the robot can reach the goal at $T$ while avoiding obstacles. At each step, the filter solves a quadratic program (QP)

$$
\begin{aligned}
\min_{\mathbf{u} \in \mathcal{U}} \quad & \|\mathbf{u} - \mathbf{u}_{\mathrm{ref}}\|^2 \\
\text{s.t.} \quad & \frac{\partial h}{\partial t} + \nabla h \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d}^*) \ge -\gamma\, h.
\end{aligned}
$$

This is the CBF-QP of lecture 3 with three changes.

- Here the CBF is time-varying. Therefore, the CBF constraint gains the term $\partial h/\partial t$.
- The dynamics include a disturbance, unlike in the lecture. We evaluate the constraint at the worst-case disturbance $\mathbf{d}^* = \arg\min_{\mathbf{d} \in \mathcal{D}} \nabla h \cdot \mathbf{f}(\mathbf{x}, \mathbf{u}, \mathbf{d})$. The function `optimal_control_and_disturbance` returns this $\mathbf{d}^*$.
- The class $\mathcal{K}_\infty$ function is the linear choice $\alpha(h) = \gamma h$ with $\gamma > 0$. The reflection below explores the effect of $\gamma$.

As in the lecture, $\mathbf{u}_{\mathrm{ref}}$ passes through unchanged when it satisfies the constraint. Where the CBF of the lecture and of tutorial 1 encodes safety alone, the reach-avoid value also encodes goal reaching, called liveness [3].

**Exercise A2.** Complete `compute_cbf_terms` and `safety_filter`. Compare the filtered and unfiltered reference under the worst disturbance, then compare the filtered control inputs with A1.

*Hints.*

- `grid.interpolate` interpolates in state only. Use it on both slices returned by the supplied `time_slices(t)` to get $h=-V$ from `V_ra` and $\nabla h=-\nabla V$ from `GRADS_ra`.
- Then interpolate in time with the weight `w` that `time_slices` also returns.
- The same two slices give $\partial h/\partial t$ as a finite difference over `DT_VALUE`; mind the sign.


In [ ]:
#@title Plot helper for the filter experiment { display-mode: "form" }
def plot_filter_experiment(X_f, h_f, U_f, X_u, h_u, gamma):
    # Map with both trajectories, the CBF h over time, and the filtered control inputs.
    _, axes = plt.subplots(1, 3, figsize=(19, 5), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
    map_axes(axes[0], title="go-to-goal controller through the reach-avoid safety filter, worst disturbance")
    plot_set(V_ra[-1], axes[0], reduce=np.min, color="#08306b", label="reach-avoid set at the full horizon, min over heading")
    for name, X, h, color in [("unfiltered", X_u, h_u, "r"), ("filtered", X_f, h_f, "b")]:
        axes[0].plot(X[0], X[1], color + "-", label=name); plot_heading(axes[0], X, every=25, color=color)
        axes[1].plot(np.arange(len(h)) * DT_SIM, h, color + "-", label=name)
    axes[0].legend(fontsize=8)
    axes[1].axhline(0, color="k", lw=0.8); axes[1].set_xlabel("time in [s]"); axes[1].set_ylabel("h along the trajectory in [m]"); axes[1].legend(); axes[1].grid(True)
    plot_inputs(axes[2], np.arange(U_f.shape[1]) * DT_SIM, U_f, f"filtered control inputs, gamma = {gamma:g}, dotted lines at the bounds")
    plt.tight_layout(); plt.show()

In [ ]:
from cvxopt import matrix, solvers
solvers.options["show_progress"] = False

def time_slices(t):
    # The two stored slices around physical time t and the weight of the later one, for linear interpolation in time.
    # Slice i lies at physical time T_RA - i * DT_VALUE.
    r = np.clip(T_RA - t, 0., T_RA)                                           # remaining time, clamped to the stored horizon
    time_idx_lo = int(np.clip(np.ceil(r / DT_VALUE), 1, len(times_ra) - 1))   # slice at a physical time in [t - DT_VALUE, t]
    time_idx_hi = time_idx_lo - 1                                             # slice at a physical time in [t, t + DT_VALUE]
    w = time_idx_lo - r / DT_VALUE                                            # a value at t is (1 - w) * value[time_idx_lo] + w * value[time_idx_hi]
    return time_idx_lo, time_idx_hi, w

def compute_cbf_terms(x, t):
    # Terms of the CBF constraint at (x, t): h, its state gradient, its partial time derivative, and the worst disturbance.
    x_clip = clip_to_grid(x)
    time_idx_lo, time_idx_hi, w = time_slices(t)
    # BEGIN SOLUTION
    return None, None, None, None   # h, state gradient (3,), partial time derivative, worst disturbance (2,)
    # END SOLUTION

def safety_filter(x, u_ref, t, gamma=0.5):
    # Solve the CBF-QP when u_ref violates the constraint.
    h, grad_h, dh_dt, d_worst = compute_cbf_terms(x, t)
    x_clip = clip_to_grid(x)
    # Drift f_0, control Jacobian G_u and disturbance Jacobian G_d at the state, so that f = f0 + Gu @ u + Gd @ d.
    f0, Gu, Gd = (np.asarray(m) for m in (dyn_reach_d.open_loop_dynamics(x_clip, 0.), dyn_reach_d.control_jacobian(x_clip, 0.), dyn_reach_d.disturbance_jacobian(x_clip, 0.)))
    u_ref = np.asarray(u_ref, dtype=float)
    # BEGIN SOLUTION
    u = None
    # END SOLUTION
    return u, d_worst, h

def go_to_goal(x, k_th=2.0):
    # Full-speed goal steering without obstacle avoidance.
    ang = math.atan2(xG[1] - x[1], xG[0] - x[0])
    return np.array([V_MAX, np.clip(k_th * wrap(ang - x[2]), -W_MAX, W_MAX)])

def rollout_reference(x_start, dt, n_steps, filtered=True, gamma=0.5):
    # Apply the filtered or unfiltered reference under the worst disturbance. Stop at goal entry.
    X = np.zeros((3, n_steps + 1)); X[:, 0] = x_start; h_traj = np.zeros(n_steps + 1); U = np.zeros((2, n_steps)); n_active = 0
    for k in range(n_steps):
        u_ref = go_to_goal(X[:, k])
        if filtered:
            u, d_worst, h_traj[k] = safety_filter(X[:, k], u_ref, k * dt, gamma=gamma)
        else:
            h_traj[k], _, _, d_worst = compute_cbf_terms(X[:, k], k * dt)
            u = u_ref
        U[:, k] = u
        n_active += int(np.linalg.norm(u - u_ref) > 1e-3)
        X[:, k + 1] = X[:, k] + dt * unicycle_f(X[:, k], u, d_worst)
        if np.linalg.norm(X[:2, k + 1] - xG[:2]) <= GOAL_RADIUS:
            h_traj[k + 1] = compute_cbf_terms(X[:, k + 1], (k + 1) * dt)[0]
            return X[:, :k + 2], h_traj[:k + 2], U[:, :k + 1], n_active / (k + 1)
    h_traj[n_steps] = compute_cbf_terms(X[:, n_steps], n_steps * dt)[0]
    return X, h_traj, U, n_active / n_steps

def run_filter_experiment(gamma=0.5):
    n_sim = int(2 * T_RA / DT_SIM)
    X_f, h_f, U_f, share = rollout_reference(x0, DT_SIM, n_sim, gamma=gamma)
    X_u, h_u, _, _ = rollout_reference(x0, DT_SIM, n_sim, filtered=False, gamma=gamma)
    for name, X, h in [("filtered", X_f, h_f), ("unfiltered", X_u, h_u)]:
        print(f"{name:10s} reference controller: min h {h.min():+.2f}, {rollout_summary(X, DT_SIM)}")
    print(f"the filter changes the control input in {100 * share:.0f} % of the steps")
    assert h_f.min() >= -0.02 and static_clearance(X_f) >= ROBOT_RADIUS - 0.02 and reached_goal(X_f) and (X_f.shape[1] - 1) * DT_SIM <= T_RA, "the filtered controller should keep h >= 0, stay clear of the obstacles and reach the goal within the horizon"

    plot_filter_experiment(X_f, h_f, U_f, X_u, h_u, gamma)
run_filter_experiment()


> **Reflection:** Change `GAMMA` in the cell below and compare the plots for `0.5`, `0.1`, and `0.01`. Then keep `GAMMA = 0.01` and refine one numerical parameter at a time using the parameter cell after C1. How do the intervention and control input smoothness change? If the QP fails, check whether refinement allows the rollout to finish. Explain your observations using the CBF condition and numerical approximation errors.
>
> | Parameter | Values to try | After running the parameter cell, rerun |
> |---|---|---|
> | `DT_SIM` | `0.05`, `0.025` s | The `GAMMA` cell below |
> | `DT_VALUE` | `0.25`, `0.1`, `0.05` s | All code cells from C3 through the `GAMMA` cell below |
> | `GRID_SHAPE` | `(71, 61, 51)`, `(141, 121, 102)` | All code cells after the parameter cell through the `GAMMA` cell below |
>
> The finer grid takes longer to solve and uses more memory.

In [ ]:
GAMMA = 0.5   # try 0.5, 0.1, 0.01
run_filter_experiment(gamma=GAMMA)

### A3. Controller design with moving obstacles

We use the same setup as the E-level, with each obstacle moving at a known constant velocity, given as `obstacle_velocities` in the cell below. In practice, a trajectory predictor such as Trajectron++ [4] could supply time-dependent obstacle positions instead.

**Exercise A3.** Extend the reach-avoid problem from C5 to moving obstacles and design a feedback controller that reaches the goal at $T$ while avoiding collisions throughout the horizon. You may adapt the HJ controller from A1, the safety filter from A2, or another approach you motivate.

1. Formulate and solve the reach-avoid problem with time-dependent obstacles. Store the values in `V_ra_moving` on `grid` at `times_ra`.
2. Implement `moving_controller(state, t)` to return the control input `[v, w]` at physical time `t`, and explain your choice.
3. Run the supplied evaluation through $T$ under the worst disturbance selected from the moving-obstacle value function.

*Hints.*

- Obstacle $i$ at physical time $t$ is `obstacles[i] + t * obstacle_velocities[i]`.
- `compute_sd_obs` from C2 gives $l_{\mathcal{O}}$ for any list of polygons, so `moving_obstacle_values(t)` is built like `g_obs` in C2 and C4 with the translated polygons.
- The supplied `moving_obstacle_postprocessor` replaces `hj.solver.static_obstacle(g_obs)` from C4 and C5.
- `GRADS_ra_moving` holds the value gradient at every node and stored slice. Evaluate it at `(state, t)` with `grid.interpolate`, on the nearest slice as in A1 or on both slices from `time_slices(t)` as in A2.
- HJ controller as in A1: `dyn_reach_d.optimal_control(state, time_dyn, grad_value)`. Safety filter as in A2: adapt `compute_cbf_terms` to `V_ra_moving` and `GRADS_ra_moving`; the worst disturbance $\mathbf{d}^*$ is `dyn_reach_d.optimal_disturbance(state, time_dyn, grad_value)`.


In [ ]:
obstacle_velocities = [np.array(v) for v in [[0.15, 0.0], [0.2089, 0.0836], [0.0557, -0.1393], [-0.5367, 0.2683], [-0.0448, 0.4478], [0.0735, 0.0147]]]   # translation velocity [vx, vy] of each polygon, in m/s, as in the E-level notebook

# BEGIN SOLUTION
def moving_obstacle_values(t):
    # t: physical time, a float.
    # Returns g = -l_O on the grid with the obstacles at their positions at time t, shape GRID_SHAPE, like g_obs in C4.
    return None
# END SOLUTION

# Checks: shape, agreement with C4 at t = 0, and signs inside an obstacle and at the start.
g0 = moving_obstacle_values(0.)
assert g0.shape == GRID_SHAPE, f"moving_obstacle_values returns shape {g0.shape}, expected {GRID_SHAPE}"
assert np.allclose(np.asarray(g0), np.asarray(g_obs)), "at t = 0 the obstacles have not moved, so g should equal g_obs from C4"
assert float(grid.interpolate(g0, jnp.array([*obstacles[0].mean(0), 0.]))) > 0 and float(grid.interpolate(g0, jnp.array(x0))) < 0

# Check plot: the zero level set of the obstacle function against the polygons at the same physical time.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, t in zip(axes, [0., T_RA]):
    map_axes(ax, title=f"moving obstacles at t = {t:.0f} s", polygons=[o + t * v for o, v in zip(obstacles, obstacle_velocities)])
    plot_set(-moving_obstacle_values(t), ax, theta=0., color="k")
plt.show()

# Shapely cannot run inside the JAX-compiled solve, so we precompute g every DT_OBS seconds and look up the nearest stored time.
DT_OBS = 0.1                                            # time spacing of the precomputed obstacle functions, in s
TIMES_OBS = np.arange(0., T_RA + DT_OBS / 2, DT_OBS)    # physical times at which g is precomputed
g_stack = jnp.stack([moving_obstacle_values(t) for t in TIMES_OBS])   # shape (len(TIMES_OBS), *GRID_SHAPE)

In [ ]:
def moving_obstacle_postprocessor(t_s, v):
    # Obstacle constraint after each solver step at solver time t_s: the moving-obstacle counterpart of hj.solver.static_obstacle(g_obs).
    k = jnp.clip(jnp.round((T_RA + t_s) / DT_OBS).astype(jnp.int32), 0, len(TIMES_OBS) - 1)   # nearest stored physical time
    return jnp.maximum(v, g_stack[k])

settings_ra_moving = hj.SolverSettings.with_accuracy("medium", value_postprocessor=moving_obstacle_postprocessor)
# BEGIN SOLUTION
terminal_values = None   # reach-avoid terminal values at T_RA, shape GRID_SHAPE
V_ra_moving = None       # solve at times_ra, shape (len(times_ra), *GRID_SHAPE)
# END SOLUTION

assert float(grid.interpolate(V_ra_moving[-1], jnp.array(x0))) <= 0, "the start state should be inside the set"
moving_frames = list(range(len(times_ra) - 1, -1, -2))                       # physical time from 0 to T, every half second
animate_sets([(f"reach-avoid set with moving obstacles, heading {math.degrees(THETA_SLICE):.0f} deg", lambda k: heading_slice(V_ra_moving[k], THETA_SLICE))], moving_frames,
             lambda k: f"t = {T_RA + times_ra[k]:.1f} s", polygons_at=lambda k: [o + (T_RA + times_ra[k]) * v for o, v in zip(obstacles, obstacle_velocities)], robot_at=lambda k: start_pose(THETA_SLICE))

In [ ]:
GRADS_ra_moving = jnp.stack([grid.grad_values(v) for v in V_ra_moving])   # grad V of the moving-obstacle value, same layout as GRADS_ra

def moving_controller(state, t):
    # Control input [v, w] at `state` and physical time t, shape (2,). Define any helper functions you need in this cell.
    # BEGIN SOLUTION
    return None
    # END SOLUTION


In [ ]:
def rollout_moving(x_start, dt):
    # Roll out moving_controller under the worst disturbance from the moving-obstacle value gradient. Does not stop at the goal.
    n = int(np.ceil(T_RA / dt))
    times = np.linspace(0., T_RA, n + 1)
    X = np.zeros((3, n + 1)); U = np.zeros((2, n)); D = np.zeros((2, n))
    X[:, 0] = x_start
    for k, t in enumerate(times[:-1]):
        u = np.asarray(moving_controller(X[:, k], t))
        assert u.shape == (2,), "moving_controller should return a control input of shape (2,)"
        U[:, k] = u
        x_clip = clip_to_grid(X[:, k])
        time_idx = int(np.argmin(np.abs(times_ra - (-T_RA + t))))                                # nearest stored slice, as in A1
        D[:, k] = np.asarray(dyn_reach_d.optimal_disturbance(x_clip, time_dyn, grid.interpolate(GRADS_ra_moving[time_idx], x_clip)))
        X[:, k + 1] = X[:, k] + (times[k + 1] - t) * unicycle_f(X[:, k], U[:, k], D[:, k])
    return X, U, D, times

X_moving, U_moving, D_moving, moving_times = rollout_moving(x0, DT_SIM)
polygons = [[o + t * v for o, v in zip(obstacles, obstacle_velocities)] for t in moving_times]
obstacle_margin = clearance(X_moving, polygons) - ROBOT_RADIUS
boundary_margin = np.min([X_moving[0], MAP_W - X_moving[0], X_moving[1], MAP_H - X_moving[1]]) - ROBOT_RADIUS
goal_error = np.linalg.norm(X_moving[:2, -1] - xG[:2])
inputs_valid = np.all((U_moving[0] >= V_MIN - 1e-6) & (U_moving[0] <= V_MAX + 1e-6)) and np.all(np.abs(U_moving[1]) <= W_MAX + 1e-6)
print(f"worst disturbance, goal error at T {goal_error:.3f} m, in goal {goal_error <= GOAL_RADIUS}, "
      f"obstacle margin {obstacle_margin:.3f} m, boundary margin {boundary_margin:.3f} m, inputs valid {inputs_valid}")
assert np.all(np.isfinite(X_moving)) and np.all(np.isfinite(U_moving))
assert np.all(np.abs(D_moving) <= D_MAX + 1e-6)
assert inputs_valid, "the controller exceeds the control input bounds"
assert goal_error <= GOAL_RADIUS, "the robot is outside the goal at T"
assert obstacle_margin >= -1e-6 and boundary_margin >= -1e-6, "a sampled state collides with an obstacle or map boundary"


In [ ]:
#@title Animate the moving-obstacle rollout { display-mode: "form" }
frame_ids = sorted(set(range(0, len(moving_times), max(1, int(0.5 / DT_SIM)))) | {len(moving_times) - 1})
animate_sets([("moving-obstacle controller, worst disturbance", lambda k: heading_slice(V_ra_moving[int(np.argmin(np.abs(times_ra - (-T_RA + moving_times[k]))))], wrap(X_moving[2, k])))],
             frame_ids, lambda k: f"t = {moving_times[k]:.2f} s",
             polygons_at=lambda k: [o + moving_times[k] * v for o, v in zip(obstacles, obstacle_velocities)],
             robot_at=lambda k: X_moving[:, k], path=X_moving)


> **Reflection:** The controller accounts for the worst disturbance acting on the robot. Does this also protect against errors in the predicted obstacle motion? How could these errors be taken into account?

## References

[1] Bansal, S., Chen, M., Herbert, S. and Tomlin, C. J. (2017). Hamilton-Jacobi reachability: A brief overview and recent advances. In *IEEE Conference on Decision and Control (CDC)*, 2242–2253.

[2] Fisac, J. F., Chen, M., Tomlin, C. J. and Sastry, S. S. (2015). Reach-avoid problems with time-varying dynamics, targets and constraints. In *Hybrid Systems: Computation and Control (HSCC)*, 11–20.

[3] Begzadić, A., Shinde, N., Tonkens, S., Hirsch, D., Ugalde, K., Yip, M., Cortés, J. and Herbert, S. (2025). Back to Base — Towards Hands-Off Learning via Safe Resets with Reach-Avoid Safety Filters. In *Learning for Dynamics and Control Conference (L4DC)*, PMLR 283, 1154–1166.

[4] Salzmann, T., Ivanovic, B., Chakravarty, P. and Pavone, M. (2020). Trajectron++: Dynamically-Feasible Trajectory Forecasting With Heterogeneous Data. In *European Conference on Computer Vision (ECCV)*.
